# YOLO-seg Training Notebook — Weed Detection on PhenoBench

**Author:** generated from spec.md
**Dataset:** PhenoBench v1.1.0
**Task:** instance segmentation (crop vs weed), spray-point estimation

---

### License Note

Ultralytics (YOLO11 / YOLOv8) is licensed under **AGPL-3.0**.
For any commercial deployment you must either publish the entire derivative work under AGPL-3.0 or obtain a commercial license from Ultralytics.
See <https://ultralytics.com/license> for details.

---

### Discrepancy Note — folder name

The spec calls the semantic-mask folder `semantic/` but the actual PhenoBench v1.1.0 directory is named **`semantics/`**.
We use `semantics/` throughout and record this in `CONFIG["SEMANTIC_DIR"]`.


## 1. Config and Imports

In [13]:
# ─── 1. CONFIG & IMPORTS ───────────────────────────────────────────────────────
import os, sys, json, random, shutil, time, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import cv2
import pandas as pd
import matplotlib
matplotlib.use("Agg")           # non-interactive backend; use 'inline' in Jupyter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import scipy
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
import sklearn
from sklearn import metrics as sk_metrics
import ultralytics
from ultralytics import YOLO

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── CONFIG ─────────────────────────────────────────────────────────────────────
CONFIG = {
    # ── paths ──────────────────────────────────────────────────────────────
    "DATASET_ROOT":  "PhenoBench-v110/PhenoBench",
    "OUTPUT_DIR":    "yolo_dataset",
    "RUNS_DIR":      "runs/segment",
    "RUN_NAME":      "phenobench_yolo11s",
    "SUMMARY_DIR":   "results",
    # ── folder names inside each split ─────────────────────────────────────
    # NOTE: spec says "semantic/" but actual PhenoBench folder is "semantics/"
    "SEMANTIC_DIR":  "semantics",
    "INSTANCES_DIR": "plant_instances",
    "IMAGES_DIR":    "images",
    # ── class mapping ──────────────────────────────────────────────────────
    # Semantic IDs (verify below): 0=soil 1=crop 2=weed 3=partial_crop 4=partial_weed
    # YOLO classes : 0=crop  1=weed   (3 → crop, 4 → weed)
    "CLASS_NAMES":   {0: "crop", 1: "weed"},
    "SEM_TO_YOLO":   {1: 0, 2: 1, 3: 0, 4: 1},
    # ── conversion ─────────────────────────────────────────────────────────
    "MIN_AREA_PX":      20,
    "CONTOUR_EPSILON":  0.75,
    # ── split ──────────────────────────────────────────────────────────────
    "TRAIN_RATIO":   0.85,   # 85% train / 15% internal-val from official train
    # ── model & training ───────────────────────────────────────────────────
    "MODEL":         "yolo11s-seg.pt",   # fallback: yolov8s-seg.pt
    "IMGSZ":         1024,
    "EPOCHS":        100,
    "PATIENCE":      30,
    "BATCH":         -1,     # auto
    "OPTIMIZER":     "auto",
    "COS_LR":        True,
    "CLOSE_MOSAIC":  10,
    "AMP":           True,
    "OVERLAP_MASK":  True,
    "RETINA_MASKS":  True,
    "WORKERS":       4,
    # ── augmentation ───────────────────────────────────────────────────────
    "HSV_H":   0.015,
    "HSV_S":   0.7,
    "HSV_V":   0.4,
    "FLIPLR":  0.5,
    "FLIPUD":  0.5,
    "SCALE":   0.4,
    "DEGREES": 10.0,
    "MOSAIC":  1.0,
    "MIXUP":   0.0,
    # ── inference operating point (updated after threshold selection) ───────
    "CONF": 0.25,
    "IOU":  0.45,
    # ── sprayer-specific  *** PLACEHOLDERS *** ──────────────────────────────
    "NOZZLE_RADIUS_PX": 10,    # set from hardware / field calibration
    "GSD_MM_PX":        None,  # ground sampling distance; errors in px until set
    # ── acceptance criteria  *** PLACEHOLDERS *** ──────────────────────────
    "TARGET_WEED_HIT_RATE_PCT":    None,  # >= X %
    "TARGET_CROP_DAMAGE_RATE_PCT": None,  # <= Y %
    "TARGET_LATENCY_MS":           None,  # <= Z ms
    "SEED": 0,
}

# ── GPU report ────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    gp = torch.cuda.get_device_properties(0)
    print(f"GPU : {gp.name}  |  VRAM : {gp.total_memory/1e9:.2f} GB")
else:
    print("WARNING: No CUDA GPU detected — training on CPU will be very slow.")

# ── Package versions ──────────────────────────────────────────────────────────
versions = dict(
    python=sys.version,
    torch=torch.__version__,
    ultralytics=ultralytics.__version__,
    opencv=cv2.__version__,
    numpy=np.__version__,
    scipy=scipy.__version__,
    pandas=pd.__version__,
    sklearn=sklearn.__version__,
    matplotlib=matplotlib.__version__,
)
for k, v in versions.items():
    print(f"  {k:<14} {v}")

Path(CONFIG["SUMMARY_DIR"]).mkdir(parents=True, exist_ok=True)
with open(f"{CONFIG['SUMMARY_DIR']}/versions.json", "w") as f:
    json.dump(versions, f, indent=2)
print("\nVersions saved.")
print("\nCONFIG loaded — keys:", list(CONFIG.keys()))


GPU : Tesla V100-SXM2-16GB  |  VRAM : 16.93 GB
  python         3.9.25 (main, Aug  6 2026, 00:00:00) 
[GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]
  torch          2.5.1+cu121
  ultralytics    8.4.187
  opencv         5.0.0
  numpy          2.0.2
  scipy          1.13.1
  pandas         2.3.3
  sklearn        1.6.1
  matplotlib     3.9.4

Versions saved.

CONFIG loaded — keys: ['DATASET_ROOT', 'OUTPUT_DIR', 'RUNS_DIR', 'RUN_NAME', 'SUMMARY_DIR', 'SEMANTIC_DIR', 'INSTANCES_DIR', 'IMAGES_DIR', 'CLASS_NAMES', 'SEM_TO_YOLO', 'MIN_AREA_PX', 'CONTOUR_EPSILON', 'TRAIN_RATIO', 'MODEL', 'IMGSZ', 'EPOCHS', 'PATIENCE', 'BATCH', 'OPTIMIZER', 'COS_LR', 'CLOSE_MOSAIC', 'AMP', 'OVERLAP_MASK', 'RETINA_MASKS', 'WORKERS', 'HSV_H', 'HSV_S', 'HSV_V', 'FLIPLR', 'FLIPUD', 'SCALE', 'DEGREES', 'MOSAIC', 'MIXUP', 'CONF', 'IOU', 'NOZZLE_RADIUS_PX', 'GSD_MM_PX', 'TARGET_WEED_HIT_RATE_PCT', 'TARGET_CROP_DAMAGE_RATE_PCT', 'TARGET_LATENCY_MS', 'SEED']


## 2. Dataset Inspection

In [14]:
# ─── 2a. File counts & semantic-ID verification ───────────────────────────────
ROOT   = Path(CONFIG["DATASET_ROOT"])
SPLITS = ["train", "val"]      # 'test' has no GT labels

def list_imgs(split):
    return sorted((ROOT / split / CONFIG["IMAGES_DIR"]).glob("*.png"))

for sp in SPLITS:
    print(f"{sp}: {len(list_imgs(sp))} images")

# ── Verify semantic IDs on 5 random training images ──────────────────────────
print("\n--- Semantic-ID verification (5 random train images) ---")
rng_imgs = random.sample(list_imgs("train"), 5)
for p in rng_imgs:
    sem = cv2.imread(str(ROOT / "train" / CONFIG["SEMANTIC_DIR"] / p.name), cv2.IMREAD_UNCHANGED)
    assert sem is not None, f"Cannot read {p}"
    print(f"  {p.name}  unique={np.unique(sem)}  dtype={sem.dtype}  shape={sem.shape}")

print("\nExpected: 0=soil  1=crop  2=weed  3=partial_crop  4=partial_weed")
print("If IDs differ, update CONFIG['SEM_TO_YOLO'] and state the discrepancy.")


train: 1407 images
val: 772 images

--- Semantic-ID verification (5 random train images) ---
  05-26_00188_P0034020.png  unique=[0 1 2 3]  dtype=uint16  shape=(1024, 1024)
  05-26_00219_P0034018.png  unique=[0 1 2 3 4]  dtype=uint16  shape=(1024, 1024)
  05-15_00102_P0030855.png  unique=[0 1 3]  dtype=uint16  shape=(1024, 1024)
  05-15_00227_P0030688.png  unique=[0 1 2]  dtype=uint16  shape=(1024, 1024)
  06-05_00055_P0037818.png  unique=[0 1 2 3]  dtype=uint16  shape=(1024, 1024)

Expected: 0=soil  1=crop  2=weed  3=partial_crop  4=partial_weed
If IDs differ, update CONFIG['SEM_TO_YOLO'] and state the discrepancy.


In [15]:
# ─── 2b. Class counts & area histogram ────────────────────────────────────────
stat        = {sp: Counter() for sp in SPLITS}
areas_all   = []
instances_per_image = {sp: [] for sp in SPLITS}

for sp in SPLITS:
    for img_path in list_imgs(sp):
        inst = cv2.imread(str(ROOT / sp / CONFIG["INSTANCES_DIR"] / img_path.name), cv2.IMREAD_UNCHANGED)
        sem  = cv2.imread(str(ROOT / sp / CONFIG["SEMANTIC_DIR"]  / img_path.name), cv2.IMREAD_UNCHANGED)
        if inst is None or sem is None:
            continue
        n_inst = 0
        for iid in np.unique(inst):
            if iid == 0:
                continue
            mask    = inst == iid
            maj_sem = Counter(sem[mask].tolist()).most_common(1)[0][0]
            yc      = CONFIG["SEM_TO_YOLO"].get(int(maj_sem), -1)
            if yc == -1:
                continue
            stat[sp][yc] += 1
            areas_all.append(int(mask.sum()))
            n_inst += 1
        instances_per_image[sp].append(n_inst)

print("Instance counts per class per split:")
for sp in SPLITS:
    tot = sum(stat[sp].values())
    for cid, cname in CONFIG["CLASS_NAMES"].items():
        n = stat[sp][cid]
        print(f"  {sp:<6}  {cname:<6}  {n:5d}  ({100*n/max(tot,1):.1f}%)")
    print(f"  {sp:<6}  TOTAL   {tot:5d}  |  avg instances/img: {np.mean(instances_per_image[sp]):.1f}")
    print()

areas_np = np.array(areas_all)
buckets  = [0, 100, 500, 2000, int(1e9)]
blabels  = ["<100 (tiny)", "100–500", "500–2000", ">2000"]
print("Area histogram (all splits, all classes):")
for lo, hi, lab in zip(buckets, buckets[1:], blabels):
    c = ((areas_np >= lo) & (areas_np < hi)).sum()
    print(f"  {lab:>15} px²:  {c:5d}  ({100*c/len(areas_np):.1f}%)")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.log10(areas_np + 1), bins=70, edgecolor="k", alpha=0.8)
ax.axvline(np.log10(CONFIG["MIN_AREA_PX"]+1), color="red", ls="--",
           label=f"MIN_AREA = {CONFIG['MIN_AREA_PX']} px²")
ax.set_xlabel("log10(area px²)")
ax.set_ylabel("count")
ax.set_title("Instance area distribution — all splits")
ax.legend()
plt.tight_layout()
plt.savefig(f"{CONFIG['SUMMARY_DIR']}/area_histogram.png", dpi=100)
plt.show()
print("Histogram saved.")


Instance counts per class per split:
  train   crop    11875  (59.3%)
  train   weed     8141  (40.7%)
  train   TOTAL   20016  |  avg instances/img: 14.2

  val     crop     6482  (62.3%)
  val     weed     3926  (37.7%)
  val     TOTAL   10408  |  avg instances/img: 13.5

Area histogram (all splits, all classes):
      <100 (tiny) px²:   2932  (9.6%)
          100–500 px²:   8956  (29.4%)
         500–2000 px²:   5926  (19.5%)
            >2000 px²:  12610  (41.4%)
Histogram saved.


## 3. Mask-to-YOLO Polygon Conversion & Split Creation

In [16]:
# ─── 3a. Conversion helpers ───────────────────────────────────────────────────

def instance_to_polygons(inst_mask, sem_mask, min_area, epsilon):
    """
    Convert one instance mask + semantic mask to a list of (yolo_class, polygon)
    tuples where polygon is an (N,2) array of normalised (x, y) coordinates.
    Returns: list[(class_id, np.ndarray shape (N,2))], dropped_count
    """
    H, W = inst_mask.shape
    records  = []
    dropped  = 0

    for iid in np.unique(inst_mask):
        if iid == 0:
            continue
        mask    = (inst_mask == iid).astype(np.uint8)
        maj_sem = Counter(sem_mask[mask == 1].tolist()).most_common(1)[0][0]
        yc      = CONFIG["SEM_TO_YOLO"].get(int(maj_sem), -1)
        if yc == -1:
            continue

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < min_area:
                dropped += 1
                continue
            approx = cv2.approxPolyDP(cnt, epsilon, closed=True)
            if len(approx) < 3:
                dropped += 1
                continue
            pts = approx.reshape(-1, 2).astype(float)
            pts[:, 0] /= W      # normalise x
            pts[:, 1] /= H      # normalise y
            records.append((yc, pts))

    return records, dropped


def write_label(label_path, records):
    """Write YOLO seg label file: one line per polygon."""
    with open(label_path, "w") as f:
        for cls_id, pts in records:
            coords = " ".join(f"{x:.6f} {y:.6f}" for x, y in pts)
            f.write(f"{cls_id} {coords}\n")


In [17]:
# ─── 3b. Per-date contiguous-block train / internal-val split ────────────────
# Filenames: MM-DD_NNNNN_PXXXXXXX.png → date = first 5 chars, frame index = NNNNN
def get_date_prefix(stem):
    return stem[:5]

def get_frame_idx(stem):
    return int(stem.split("_")[1])

BUFFER_FRAMES = 3   # frames dropped at each train/val boundary to avoid overlap

train_imgs  = list_imgs("train")
date_groups = {}
for p in train_imgs:
    date_groups.setdefault(get_date_prefix(p.stem), []).append(p)
all_dates = sorted(date_groups)

int_train_imgs, int_val_imgs, dropped_buffer = [], [], []
for dp in all_dates:
    imgs  = sorted(date_groups[dp], key=lambda p: get_frame_idx(p.stem))
    n_val = max(1, round(len(imgs) * (1 - CONFIG["TRAIN_RATIO"])))
    cut   = len(imgs) - n_val                       # val = last 15% of frames
    int_train_imgs += imgs[:max(0, cut - BUFFER_FRAMES)]
    dropped_buffer += imgs[max(0, cut - BUFFER_FRAMES):cut]
    int_val_imgs   += imgs[cut:]
    print(f"  {dp}: {len(imgs)} imgs → train {max(0, cut - BUFFER_FRAMES)}, "
          f"buffer {min(BUFFER_FRAMES, cut)}, val {n_val}")

test_imgs = list_imgs("val")   # official val = held-out test

assert len(int_val_imgs) > 0,   "Internal val is empty — check split logic"
assert len(int_train_imgs) > 0, "Internal train is empty — check split logic"

print(f"\ninternal train : {len(int_train_imgs)}")
print(f"internal val   : {len(int_val_imgs)}")
print(f"buffer dropped : {len(dropped_buffer)}")
print(f"held-out test  : {len(test_imgs)}")

OUT = Path(CONFIG["OUTPUT_DIR"])
(OUT / "splits").mkdir(parents=True, exist_ok=True)
for name, lst in [("train", int_train_imgs), ("val", int_val_imgs), ("test", test_imgs)]:
    (OUT / "splits" / f"{name}.txt").write_text("".join(p.name + "\n" for p in lst))

  05-15: 613 imgs → train 518, buffer 3, val 92
  05-26: 396 imgs → train 334, buffer 3, val 59
  06-05: 398 imgs → train 335, buffer 3, val 60

internal train : 1187
internal val   : 211
buffer dropped : 9
held-out test  : 772


In [18]:
# Run once before cell 3c
for sp in ["train", "val", "test"]:
    shutil.rmtree(OUT / sp, ignore_errors=True)

In [19]:
# ─── 3c. Convert labels and create YOLO folder structure ─────────────────────
# Structure: yolo_dataset/{train,val,test}/{images,labels}/

dropped_total  = 0
instance_total = 0

split_cfg = {
    "train": (int_train_imgs, "train"),
    "val":   (int_val_imgs,   "train"),     # both come from official train
    "test":  (test_imgs,      "val"),       # official val split
}

for yolo_split, (img_list, src_split) in split_cfg.items():
    img_out = OUT / yolo_split / "images"
    lbl_out = OUT / yolo_split / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_path in img_list:
        # ── image: symlink (Windows mklink) or copy ─────────────────────
        dst_img = img_out / img_path.name
        if not dst_img.exists():
            try:
                dst_img.symlink_to(img_path.resolve())
            except (OSError, NotImplementedError):
                shutil.copy2(img_path, dst_img)

        # ── label ────────────────────────────────────────────────────────
        inst_path = ROOT / src_split / CONFIG["INSTANCES_DIR"] / img_path.name
        sem_path  = ROOT / src_split / CONFIG["SEMANTIC_DIR"]  / img_path.name
        lbl_path  = lbl_out / (img_path.stem + ".txt")

        if not inst_path.exists() or not sem_path.exists():
            # test split has no GT — write empty label
            lbl_path.write_text("")
            continue

        inst = cv2.imread(str(inst_path), cv2.IMREAD_UNCHANGED)
        sem  = cv2.imread(str(sem_path),  cv2.IMREAD_UNCHANGED)
        assert inst is not None and sem is not None

        records, dropped = instance_to_polygons(
            inst, sem, CONFIG["MIN_AREA_PX"], CONFIG["CONTOUR_EPSILON"]
        )
        write_label(lbl_path, records)
        dropped_total  += dropped
        instance_total += len(records)

    print(f"[{yolo_split}] {len(img_list)} images processed")

print(f"\nTotal polygons written : {instance_total}")
print(f"Total fragments dropped: {dropped_total}  "
      f"(area < {CONFIG['MIN_AREA_PX']} px or < 3 points)")


[train] 1187 images processed
[val] 211 images processed
[test] 772 images processed

Total polygons written : 35770
Total fragments dropped: 2660  (area < 20 px or < 3 points)


In [20]:
# ─── 3d. Sanity check ────────────────────────────────────────────────────────
def names_in(split):
    return set((OUT / "splits" / f"{split}.txt").read_text().split())

tr, iv, te = names_in("train"), names_in("val"), names_in("test")
assert not (tr & iv), f"train/val share {len(tr & iv)} files"
assert not (tr & te), f"train/test share {len(tr & te)} files"
assert not (iv & te), f"val/test share {len(iv & te)} files"

# Each split's image folder must match its split list exactly (catches stale symlinks)
for sp, names in [("train", tr), ("val", iv), ("test", te)]:
    on_disk = {p.name for p in (OUT / sp / "images").glob("*.png")}
    assert on_disk == names, f"{sp}/images doesn't match splits/{sp}.txt"

print("Dates in train:", sorted({n[:5] for n in tr}))
print("Dates in val  :", sorted({n[:5] for n in iv}))
print("No file overlap between splits ✓")

Dates in train: ['05-15', '05-26', '06-05']
Dates in val  : ['05-15', '05-26', '06-05']
No file overlap between splits ✓


## 4. Label Visual Check

In [21]:
# ─── 4. Overlay 8 random converted labels on their images ─────────────────────
COLORS = {0: (0, 255, 0), 1: (0, 0, 255)}   # BGR: crop=green, weed=red

def draw_label_overlay(img_path, lbl_path, alpha=0.4):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    H, W = img.shape[:2]
    overlay = img.copy()
    if not lbl_path.exists() or lbl_path.stat().st_size == 0:
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.split()
        cls_id = int(parts[0])
        coords = list(map(float, parts[1:]))
        pts = np.array([[coords[i]*W, coords[i+1]*H]
                        for i in range(0, len(coords), 2)], dtype=np.int32)
        color = COLORS.get(cls_id, (128, 128, 128))
        cv2.fillPoly(overlay, [pts], color)
    blended = cv2.addWeighted(img, 1-alpha, overlay, alpha, 0)
    return cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)

# pick 8 images from internal train that have non-empty labels
train_label_dir = OUT / "train" / "labels"
nonempty = [p for p in sorted(train_label_dir.glob("*.txt")) if p.stat().st_size > 0]
sample8  = random.sample(nonempty, min(8, len(nonempty)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, lbl_path in zip(axes.flat, sample8):
    img_path = OUT / "train" / "images" / lbl_path.with_suffix(".png").name
    vis = draw_label_overlay(img_path, lbl_path)
    if vis is not None:
        ax.imshow(vis)
    ax.set_title(lbl_path.stem[:30], fontsize=7)
    ax.axis("off")

# legend
green_patch = mpatches.Patch(color=(0,1,0), label="crop (0)")
red_patch   = mpatches.Patch(color=(1,0,0), label="weed (1)")
fig.legend(handles=[green_patch, red_patch], loc="lower center", ncol=2, fontsize=12)
plt.suptitle("Label visual check — 8 random training images", fontsize=14)
plt.tight_layout()
plt.savefig(f"{CONFIG['SUMMARY_DIR']}/label_visual_check.png", dpi=80)
plt.show()
print("Visual check saved.")


Visual check saved.


## 5. dataset.yaml Generation

In [22]:
# ─── 5. Write dataset.yaml ────────────────────────────────────────────────────
import yaml

yaml_path = OUT / "dataset.yaml"
ds_yaml = {
    "path":  str(OUT.resolve()),
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "names": CONFIG["CLASS_NAMES"],    # {0: "crop", 1: "weed"}
}
with open(yaml_path, "w") as f:
    yaml.dump(ds_yaml, f, default_flow_style=False, sort_keys=False)

print("dataset.yaml written to:", yaml_path)
print(yaml_path.read_text())


dataset.yaml written to: yolo_dataset/dataset.yaml
path: /arc/project/st-sielmann-1/weed_seg/yolo_dataset
train: train/images
val: val/images
test: test/images
names:
  0: crop
  1: weed



## 6. Training

> **Note:** Training `yolo11s-seg.pt` at `imgsz=1024` requires a GPU with ≥ 8 GB VRAM.
> With `batch=-1` Ultralytics auto-selects the batch size.
> Set `batch=2` or `batch=4` manually if you get OOM errors.


In [23]:
# ─── 6. TRAIN ─────────────────────────────────────────────────────────────────
from ultralytics import YOLO

# Try yolo11s-seg; fall back to yolov8s-seg if not available
try:
    model = YOLO(CONFIG["MODEL"])
    print(f"Loaded {CONFIG['MODEL']}")
except Exception:
    fallback = "yolov8s-seg.pt"
    print(f"Could not load {CONFIG['MODEL']}, falling back to {fallback}")
    model = YOLO(fallback)
    CONFIG["MODEL"] = fallback

results = model.train(
    data      = str(yaml_path),
    epochs    = CONFIG["EPOCHS"],
    patience  = CONFIG["PATIENCE"],
    imgsz     = CONFIG["IMGSZ"],
    batch     = CONFIG["BATCH"],
    optimizer = CONFIG["OPTIMIZER"],
    cos_lr    = CONFIG["COS_LR"],
    close_mosaic = CONFIG["CLOSE_MOSAIC"],
    amp       = CONFIG["AMP"],
    overlap_mask = CONFIG["OVERLAP_MASK"],
    retina_masks = CONFIG["RETINA_MASKS"],
    workers   = CONFIG["WORKERS"],
    seed      = CONFIG["SEED"],
    deterministic = True,
    name      = CONFIG["RUN_NAME"],
    project   = CONFIG["RUNS_DIR"],
    # augmentation
    hsv_h    = CONFIG["HSV_H"],
    hsv_s    = CONFIG["HSV_S"],
    hsv_v    = CONFIG["HSV_V"],
    fliplr   = CONFIG["FLIPLR"],
    flipud   = CONFIG["FLIPUD"],
    scale    = CONFIG["SCALE"],
    degrees  = CONFIG["DEGREES"],
    mosaic   = CONFIG["MOSAIC"],
    mixup    = CONFIG["MIXUP"],
    # blur     = 0.0,
    verbose  = True,
)

RUN_DIR = Path(results.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
print(f"\nTraining done.  Run dir : {RUN_DIR}")
print(f"Best weights  : {BEST_PT}")


Loaded yolo11s-seg.pt
Ultralytics 8.4.187 🚀 Python-3.9.25 torch-2.5.1+cu121 CUDA:0 (Tesla V100-SXM2-16GB, 16144MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=phenobench_yolo11s-2, nbs=

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



## 7. Training Curves & Standard Validation (Internal Val)

In [24]:
# ─── 7a. Training curves ─────────────────────────────────────────────────────
csv_path = RUN_DIR / "results.csv"
assert csv_path.exists(), f"results.csv not found at {csv_path}"

df_train = pd.read_csv(csv_path)
df_train.columns = df_train.columns.str.strip()
print("Columns:", list(df_train.columns))
print(df_train.tail(5))

# Plot losses and mAP
loss_cols = [c for c in df_train.columns if "loss" in c.lower()]
map_cols  = [c for c in df_train.columns if "map" in c.lower()]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for col in loss_cols:
    axes[0].plot(df_train["epoch"], df_train[col], label=col)
axes[0].set_title("Training / Val losses")
axes[0].set_xlabel("epoch")
axes[0].legend(fontsize=7)

for col in map_cols:
    axes[1].plot(df_train["epoch"], df_train[col], label=col)
axes[1].set_title("mAP over epochs")
axes[1].set_xlabel("epoch")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig(f"{CONFIG['SUMMARY_DIR']}/training_curves.png", dpi=100)
plt.show()
print("Training curves saved.")


Columns: ['epoch', 'time', 'train/box_loss', 'train/seg_loss', 'train/cls_loss', 'train/dfl_loss', 'train/sem_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(M)', 'metrics/recall(M)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)', 'val/box_loss', 'val/seg_loss', 'val/cls_loss', 'val/dfl_loss', 'val/sem_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']
    epoch     time  train/box_loss  train/seg_loss  train/cls_loss  \
95     96  3307.80         0.57681         1.08604         0.36736   
96     97  3333.00         0.57915         1.08554         0.36750   
97     98  3358.28         0.57841         1.08345         0.36691   
98     99  3383.63         0.57744         1.08730         0.36653   
99    100  3409.44         0.57651         1.07982         0.36767   

    train/dfl_loss  train/sem_loss  metrics/precision(B)  metrics/recall(B)  \
95         0.84445               0               0.83666            0.72184   
96         0.84392

In [25]:
# ─── 7b. Standard validation on internal val ─────────────────────────────────
model_best = YOLO(str(BEST_PT))

val_results = model_best.val(
    data    = str(yaml_path),
    split   = "val",
    imgsz   = CONFIG["IMGSZ"],
    conf    = CONFIG["CONF"],
    iou     = CONFIG["IOU"],
    retina_masks = CONFIG["RETINA_MASKS"],
    verbose = True,
)

print("\n=== Internal Val — Box ===")
print(f"  mAP@0.5       : {val_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95  : {val_results.box.map:.4f}")

print("\n=== Internal Val — Mask (PRIMARY METRIC) ===")
print(f"  mAP@0.5       : {val_results.seg.map50:.4f}")
print(f"  mAP@0.5:0.95  : {val_results.seg.map:.4f}")

print("\nPer-class mask mAP@0.5:")
for i, name in CONFIG["CLASS_NAMES"].items():
    print(f"  {name}: {val_results.seg.ap50[i]:.4f}")

# Save standard metrics
std_metrics = {
    "split": "internal_val",
    "box_map50": float(val_results.box.map50),
    "box_map":   float(val_results.box.map),
    "seg_map50": float(val_results.seg.map50),
    "seg_map":   float(val_results.seg.map),
    "seg_map50_per_class": {CONFIG["CLASS_NAMES"][i]: float(val_results.seg.ap50[i])
                             for i in range(len(CONFIG["CLASS_NAMES"]))},
    "precision": float(val_results.seg.p.mean()),
    "recall":    float(val_results.seg.r.mean()),
}
with open(f"{CONFIG['SUMMARY_DIR']}/std_metrics_val.json", "w") as f:
    json.dump(std_metrics, f, indent=2)
print("\nStandard val metrics saved.")


Ultralytics 8.4.187 🚀 Python-3.9.25 torch-2.5.1+cu121 CUDA:0 (Tesla V100-SXM2-16GB, 16144MiB)
YOLO11s-seg summary (fused): 113 layers, 10,067,590 parameters, 0 gradients, 32.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 893.6±116.7 MB/s, size: 2324.9 KB)
val: Scanning /arc/project/st-sielmann-1/weed_seg/yolo_dataset/val/labels.cache... 211 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 211/211 7.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 1.7it/s 8.2s0.3s
                   all        211       3860      0.851      0.716       0.73      0.561      0.836      0.696      0.708      0.441
                  crop        211       2297       0.91      0.831      0.842      0.711      0.891      0.811       0.82      0.609
                  weed        201       1563      0.793      0.601      0.617      0.411      0.781      0.582      0.596      0.2

In [26]:
# ─── 7c. Curve plots (PR, F1, P-conf, R-conf, confusion matrix) ──────────────
# Ultralytics saves these automatically under the run directory.
curve_files = list(RUN_DIR.glob("*.png"))
print("Curve/plot files generated by Ultralytics:")
for f in sorted(curve_files):
    print(" ", f.name)
print("\nConfusion matrix and per-class curves are in:", RUN_DIR)


Curve/plot files generated by Ultralytics:
  BoxF1_curve.png
  BoxPR_curve.png
  BoxP_curve.png
  BoxR_curve.png
  MaskF1_curve.png
  MaskPR_curve.png
  MaskP_curve.png
  MaskR_curve.png
  confusion_matrix.png
  confusion_matrix_normalized.png
  results.png

Confusion matrix and per-class curves are in: /arc/project/st-sielmann-1/weed_seg/runs/segment/runs/segment/phenobench_yolo11s-2


## 8. Threshold Selection from F1 Curve

For a sprayer application we favour **high weed recall** while keeping the crop-damage rate low,
rather than naively picking the F1 maximum.
The chosen threshold is stored in `CONFIG["CONF"]`.


In [27]:
# ─── 8. Choose operating-point confidence threshold ───────────────────────────
# Ultralytics saves F1_curve.csv or encodes it in the results.
# We re-run val with save_json to extract per-threshold metrics when available.
# Fallback: use val_results.seg curves directly.

try:
    f1_csv = RUN_DIR / "F1_curve.csv"
    if f1_csv.exists():
        df_f1 = pd.read_csv(f1_csv)
        print("F1 curve columns:", list(df_f1.columns))
    else:
        # Use curves stored on the validator object
        # val_results.seg.curves_results: [px, py, ap, f1, conf]
        cr = val_results.seg.curves_results
        px = cr[0]   # confidence thresholds
        f1 = cr[3]   # F1 per class (shape: n_thresholds x n_classes)
        df_f1 = pd.DataFrame({"conf": px})
        for i, name in CONFIG["CLASS_NAMES"].items():
            df_f1[f"f1_{name}"] = f1[:, i] if f1.ndim > 1 else f1
        print("F1 curve built from curves_results")

    # ── Strategy: pick threshold for high weed recall (>=0.80) while maximising
    # overall F1. This is a conservative sprayer policy.
    # We look at weed-class F1 first, then overall mean F1 as tiebreaker.
    weed_col = "f1_weed" if "f1_weed" in df_f1.columns else df_f1.columns[-1]
    mean_f1  = df_f1[[c for c in df_f1.columns if c.startswith("f1_")]].mean(axis=1)

    # Weed recall constraint: find thresholds where weed-F1 >= 90% of its max
    weed_f1    = df_f1[weed_col].values
    threshold  = 0.9 * weed_f1.max()
    candidates = df_f1["conf"][weed_f1 >= threshold]

    # Among those candidates pick highest mean F1
    best_idx  = mean_f1[weed_f1 >= threshold].idxmax()
    BEST_CONF = float(df_f1.loc[best_idx, "conf"])

    CONFIG["CONF"] = BEST_CONF
    print(f"\nChosen confidence threshold : {BEST_CONF:.3f}")
    print(f"  weed F1 @ threshold        : {weed_f1[best_idx]:.4f}")
    print(f"  mean F1 @ threshold        : {mean_f1[best_idx]:.4f}")
    print("Reasoning: threshold set where weed-F1 >= 90% of its peak, maximising mean F1.")
    print("This favours weed recall for the sprayer while controlling false positives.")

except Exception as exc:
    print(f"Could not parse F1 curve automatically ({exc}).")
    print("Using default CONF =", CONFIG["CONF"])
    print("Inspect RUN_DIR F1_curve plots manually and update CONFIG['CONF'].")

print(f"\nFinal CONFIG['CONF'] = {CONFIG['CONF']:.3f},  CONFIG['IOU'] = {CONFIG['IOU']}")


Could not parse F1 curve automatically ('list' object has no attribute 'ndim').
Using default CONF = 0.25
Inspect RUN_DIR F1_curve plots manually and update CONFIG['CONF'].

Final CONFIG['CONF'] = 0.250,  CONFIG['IOU'] = 0.45


## 9. Final Evaluation on Held-Out Test Split

In [28]:
# ─── 9. Standard metrics on held-out test (official val split) ────────────────
test_results = model_best.val(
    data    = str(yaml_path),
    split   = "test",
    imgsz   = CONFIG["IMGSZ"],
    conf    = CONFIG["CONF"],
    iou     = CONFIG["IOU"],
    retina_masks = CONFIG["RETINA_MASKS"],
    verbose = True,
)

print("\n=== TEST — Mask (PRIMARY METRIC) ===")
print(f"  mAP@0.5       : {test_results.seg.map50:.4f}")
print(f"  mAP@0.5:0.95  : {test_results.seg.map:.4f}")
print("Per-class:")
for i, name in CONFIG["CLASS_NAMES"].items():
    ap50 = float(test_results.seg.ap50[i])
    print(f"  {name}: AP@0.5 = {ap50:.4f}")

test_std_metrics = {
    "split": "held_out_test",
    "conf":  CONFIG["CONF"],
    "iou":   CONFIG["IOU"],
    "box_map50": float(test_results.box.map50),
    "box_map":   float(test_results.box.map),
    "seg_map50": float(test_results.seg.map50),
    "seg_map":   float(test_results.seg.map),
    "precision": float(test_results.seg.p.mean()),
    "recall":    float(test_results.seg.r.mean()),
    "per_class": {CONFIG["CLASS_NAMES"][i]: {
                     "ap50": float(test_results.seg.ap50[i]),
                     "precision": float(test_results.seg.p[i]),
                     "recall":    float(test_results.seg.r[i]),
                  } for i in range(len(CONFIG["CLASS_NAMES"]))},
}
with open(f"{CONFIG['SUMMARY_DIR']}/std_metrics_test.json", "w") as f:
    json.dump(test_std_metrics, f, indent=2)
print("\nTest standard metrics saved.")


Ultralytics 8.4.187 🚀 Python-3.9.25 torch-2.5.1+cu121 CUDA:0 (Tesla V100-SXM2-16GB, 16144MiB)
YOLO11s-seg summary (fused): 113 layers, 10,067,590 parameters, 0 gradients, 32.9 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 48.4±10.3 MB/s, size: 2430.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /arc/project/st-sielmann-1/weed_seg/yolo_dataset/test/labels... 772 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 772/772 98.2it/s 7.9s<0.0s
val: New cache created: /arc/project/st-sielmann-1/weed_seg/yolo_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 2.7it/s 18.1s0.3s
                   all        772      12082      0.824      0.777      0.776      0.601      0.824      0.743      0.749      0.479
                  

## 10. Task-Specific Metrics (Sprayer Evaluation on Held-Out Test)

In [29]:
# ─── 10a. Helpers: spray point via distance-transform maximum ─────────────────

def spray_point(binary_mask):
    """Return (x, y) pixel = deepest interior point (distance-transform max)."""
    dt = ndimage.distance_transform_edt(binary_mask)
    idx = np.unravel_index(dt.argmax(), dt.shape)
    return int(idx[1]), int(idx[0])   # (x, y)


def mask_iou(mask_a, mask_b):
    inter = (mask_a & mask_b).sum()
    union = (mask_a | mask_b).sum()
    return inter / union if union > 0 else 0.0


def rle_to_mask(rle, shape):
    """Convert ultralytics RLE-ish mask (numpy array) to boolean 2-D mask."""
    return rle.astype(bool)


In [30]:
# ─── 10b. Run inference on test images and collect per-image results ──────────
test_img_dir   = OUT / "test" / "images"
test_label_dir = OUT / "test" / "labels"
test_img_list  = sorted(test_img_dir.glob("*.png"))

# Also need GT from official val split
GT_ROOT   = ROOT / "val"
GSD       = CONFIG["GSD_MM_PX"]
NR        = CONFIG["NOZZLE_RADIUS_PX"]
MIN_AREA  = CONFIG["MIN_AREA_PX"]

records_list = []      # one row per GT weed instance
seg_maps_pred = []     # for pixel-level IoU/Dice
seg_maps_gt   = []

model_inf = YOLO(str(BEST_PT))

for img_path in test_img_list:
    fname    = img_path.name
    inst_path = GT_ROOT / CONFIG["INSTANCES_DIR"] / fname
    sem_path  = GT_ROOT / CONFIG["SEMANTIC_DIR"]  / fname

    if not inst_path.exists() or not sem_path.exists():
        continue   # skip if GT missing

    gt_inst = cv2.imread(str(inst_path), cv2.IMREAD_UNCHANGED)
    gt_sem  = cv2.imread(str(sem_path),  cv2.IMREAD_UNCHANGED)
    H, W    = gt_inst.shape

    # ── GT weed instances ────────────────────────────────────────────────
    gt_weeds = {}    # {inst_id: binary_mask}
    for iid in np.unique(gt_inst):
        if iid == 0:
            continue
        mask    = gt_inst == iid
        maj_sem = Counter(gt_sem[mask].tolist()).most_common(1)[0][0]
        yc      = CONFIG["SEM_TO_YOLO"].get(int(maj_sem), -1)
        if yc == 1:      # weed only
            gt_weeds[iid] = mask

    # ── Inference ────────────────────────────────────────────────────────
    preds = model_inf.predict(
        str(img_path), imgsz=CONFIG["IMGSZ"],
        conf=CONFIG["CONF"], iou=CONFIG["IOU"],
        retina_masks=True, verbose=False,
    )[0]

    # ── GT semantic map for pixel-level metrics ───────────────────────────
    gt_semantic = np.zeros((H, W), dtype=np.int32)   # 0=bg, 1=crop, 2=weed
    for iid in np.unique(gt_inst):
        if iid == 0:
            continue
        mask    = gt_inst == iid
        maj_sem = Counter(gt_sem[mask].tolist()).most_common(1)[0][0]
        if maj_sem == 1 or maj_sem == 3:
            gt_semantic[mask] = 1
        elif maj_sem == 2 or maj_sem == 4:
            gt_semantic[mask] = 2

    # ── Predicted semantic map ────────────────────────────────────────────
    pred_semantic = np.zeros((H, W), dtype=np.int32)
    pred_weed_masks  = []

    if preds.masks is not None and len(preds.masks) > 0:
        for i, cls_tensor in enumerate(preds.boxes.cls):
            cls = int(cls_tensor.item())
            pmask = (preds.masks.data[i].cpu().numpy() > 0.5)
            # resize to original if needed
            if pmask.shape != (H, W):
                pmask = cv2.resize(pmask.astype(np.uint8), (W, H),
                                   interpolation=cv2.INTER_NEAREST).astype(bool)
            pred_semantic[pmask] = cls + 1   # 1=crop, 2=weed (last wins)
            if cls == 1:
                pred_weed_masks.append(pmask)

    seg_maps_gt.append(gt_semantic)
    seg_maps_pred.append(pred_semantic)

    if not gt_weeds:
        continue   # image has no GT weeds

    # ── Hungarian matching: pred weeds ↔ GT weeds by mask IoU ────────────
    gt_ids   = list(gt_weeds.keys())
    n_gt     = len(gt_ids)
    n_pred   = len(pred_weed_masks)

    if n_pred == 0:
        for iid in gt_ids:
            records_list.append(dict(
                image=fname, gt_id=iid,
                matched=False, iou=0.0,
                gt_area=int(gt_weeds[iid].sum()),
                spray_err_px=np.nan, spray_err_mm=np.nan,
                hit_in_mask=False, hit_in_nozzle=False,
            ))
        continue

    cost = np.zeros((n_gt, n_pred))
    for gi, gid in enumerate(gt_ids):
        for pi, pm in enumerate(pred_weed_masks):
            cost[gi, pi] = 1.0 - mask_iou(gt_weeds[gid], pm)

    row_ind, col_ind = linear_sum_assignment(cost)

    matched_pred = set()
    for gi, pi in zip(row_ind, col_ind):
        iou_val = 1.0 - cost[gi, pi]
        gid     = gt_ids[gi]
        gm      = gt_weeds[gid]
        pm      = pred_weed_masks[pi]
        matched = iou_val >= 0.5

        gt_sp   = spray_point(gm)
        pred_sp = spray_point(pm) if matched else (None, None)
        err_px  = float(np.hypot(pred_sp[0]-gt_sp[0], pred_sp[1]-gt_sp[1])) if matched else np.nan
        err_mm  = err_px * GSD if (matched and GSD) else np.nan
        hit_mask   = bool(gm[pred_sp[1], pred_sp[0]]) if matched else False
        hit_nozzle = err_px <= NR if matched else False

        matched_pred.add(pi)
        records_list.append(dict(
            image=fname, gt_id=gid,
            matched=matched, iou=iou_val,
            gt_area=int(gm.sum()),
            spray_err_px=err_px, spray_err_mm=err_mm,
            hit_in_mask=hit_mask, hit_in_nozzle=hit_nozzle,
        ))

    # unmatched GT
    unmatched_gt = set(range(n_gt)) - set(row_ind)
    for gi in unmatched_gt:
        gid = gt_ids[gi]
        records_list.append(dict(
            image=fname, gt_id=gid,
            matched=False, iou=0.0,
            gt_area=int(gt_weeds[gid].sum()),
            spray_err_px=np.nan, spray_err_mm=np.nan,
            hit_in_mask=False, hit_in_nozzle=False,
        ))

print(f"Processed {len(seg_maps_gt)} test images.")
print(f"Total GT weed instances: {len(records_list)}")


Processed 772 test images.
Total GT weed instances: 3926


In [ ]:
# ─── 10c. Compute task-specific metrics ──────────────────────────────────────
df_rec = pd.DataFrame(records_list)
df_rec.to_csv(f"{CONFIG['SUMMARY_DIR']}/spray_records.csv", index=False)

matched    = df_rec["matched"]
err_px     = df_rec.loc[matched, "spray_err_px"].dropna()
err_mm     = df_rec.loc[matched, "spray_err_mm"].dropna()
areas      = df_rec["gt_area"]

# ── Center error ─────────────────────────────────────────────────────────────
center_err = {}
center_err["mean_px"]   = float(err_px.mean())   if len(err_px) > 0 else np.nan
center_err["median_px"] = float(err_px.median()) if len(err_px) > 0 else np.nan
center_err["p90_px"]    = float(err_px.quantile(0.9)) if len(err_px) > 0 else np.nan
if GSD:
    center_err["mean_mm"]   = float(err_mm.mean())
    center_err["median_mm"] = float(err_mm.median())
    center_err["p90_mm"]    = float(err_mm.quantile(0.9))

# ── Hit rates ─────────────────────────────────────────────────────────────────
n_gt_total     = len(df_rec)
hit_rate_mask  = df_rec["hit_in_mask"].sum()   / n_gt_total * 100
hit_rate_nozzle= df_rec["hit_in_nozzle"].sum() / n_gt_total * 100

# ── Weed recall by size bucket ────────────────────────────────────────────────
size_recall = {}
for lo, hi, lab in zip([0,100,500,2000],[100,500,2000,int(1e9)],
                       ["<100","100-500","500-2000",">2000"]):
    subset = df_rec[(areas >= lo) & (areas < hi)]
    if len(subset) == 0:
        size_recall[lab] = np.nan
    else:
        size_recall[lab] = float(subset["matched"].sum() / len(subset) * 100)

# ── Crop-damage and false-spray rate ─────────────────────────────────────────
# Re-scan predicted weed spray points against GT semantic maps
n_pred_weed_total = 0
crop_damage_count = 0
false_spray_count = 0

for img_path in test_img_list:
    fname = img_path.name
    inst_path = GT_ROOT / CONFIG["INSTANCES_DIR"] / fname
    sem_path  = GT_ROOT / CONFIG["SEMANTIC_DIR"]  / fname
    if not inst_path.exists():
        continue
    gt_inst = cv2.imread(str(inst_path), cv2.IMREAD_UNCHANGED)
    gt_sem  = cv2.imread(str(sem_path),  cv2.IMREAD_UNCHANGED)
    H, W    = gt_inst.shape

    gt_crop_mask = np.isin(gt_sem, [1, 3])    # crop pixels
    gt_plant_mask= np.isin(gt_sem, [1, 2, 3, 4])  # any plant

    preds = model_inf.predict(
        str(img_path), imgsz=CONFIG["IMGSZ"],
        conf=CONFIG["CONF"], iou=CONFIG["IOU"],
        retina_masks=True, verbose=False,
    )[0]

    if preds.masks is None:
        continue

    for i, cls_tensor in enumerate(preds.boxes.cls):
        if int(cls_tensor.item()) != 1:
            continue   # only predicted weeds
        pmask = (preds.masks.data[i].cpu().numpy() > 0.5)
        if pmask.shape != (H, W):
            pmask = cv2.resize(pmask.astype(np.uint8), (W, H),
                               interpolation=cv2.INTER_NEAREST).astype(bool)
        sp = spray_point(pmask)
        n_pred_weed_total += 1
        x, y = sp
        if gt_crop_mask[y, x]:
            crop_damage_count += 1
        elif not gt_plant_mask[y, x]:
            false_spray_count += 1

crop_damage_rate = crop_damage_count / max(n_pred_weed_total, 1) * 100
false_spray_rate = false_spray_count / max(n_pred_weed_total, 1) * 100

# ── Pixel-level IoU and Dice ──────────────────────────────────────────────────
gt_flat   = np.concatenate([m.ravel() for m in seg_maps_gt])
pred_flat = np.concatenate([m.ravel() for m in seg_maps_pred])

def pix_iou_dice(gt, pred, cls):
    g = (gt == cls)
    p = (pred == cls)
    inter = (g & p).sum()
    union = (g | p).sum()
    iou_val  = inter / max(union, 1)
    dice_val = 2 * inter / max(g.sum() + p.sum(), 1)
    return float(iou_val), float(dice_val)

crop_iou,  crop_dice  = pix_iou_dice(gt_flat, pred_flat, 1)
weed_iou,  weed_dice  = pix_iou_dice(gt_flat, pred_flat, 2)

# ── Summary table ─────────────────────────────────────────────────────────────
task_metrics = {
    "center_error": center_err,
    "hit_rate_in_mask_pct":    hit_rate_mask,
    "hit_rate_in_nozzle_pct":  hit_rate_nozzle,
    "nozzle_radius_px":        NR,
    "weed_recall_by_size_pct": size_recall,
    "crop_damage_rate_pct":    crop_damage_rate,
    "false_spray_rate_pct":    false_spray_rate,
    "pixel_iou_crop":          crop_iou,
    "pixel_dice_crop":         crop_dice,
    "pixel_iou_weed":          weed_iou,
    "pixel_dice_weed":         weed_dice,
    "n_gt_weeds":              n_gt_total,
    "n_pred_weeds":            n_pred_weed_total,
}

with open(f"{CONFIG['SUMMARY_DIR']}/task_metrics.json", "w") as f:
    json.dump(task_metrics, f, indent=2, default=str)

# Pretty table
rows = [
    ("Hit rate — predicted point inside GT mask", f"{hit_rate_mask:.1f} %"),
    (f"Hit rate — within {NR} px of GT spray point", f"{hit_rate_nozzle:.1f} %"),
    ("Center error — mean",   f"{center_err.get('mean_px', 'n/a'):.1f} px"),
    ("Center error — median", f"{center_err.get('median_px', 'n/a'):.1f} px"),
    ("Center error — P90",    f"{center_err.get('p90_px', 'n/a'):.1f} px"),
    ("Crop-damage rate",       f"{crop_damage_rate:.1f} %"),
    ("False-spray rate",       f"{false_spray_rate:.1f} %"),
    ("Pixel IoU — crop",       f"{crop_iou:.4f}"),
    ("Pixel IoU — weed",       f"{weed_iou:.4f}"),
    ("Pixel Dice — crop",      f"{crop_dice:.4f}"),
    ("Pixel Dice — weed",      f"{weed_dice:.4f}"),
]
for lab, lab2 in [("Recall <100 px",    size_recall.get("<100","n/a")),
                   ("Recall 100-500 px", size_recall.get("100-500","n/a")),
                   ("Recall 500-2000 px",size_recall.get("500-2000","n/a")),
                   ("Recall >2000 px",   size_recall.get(">2000","n/a"))]:
    val_str = f"{lab2:.1f} %" if isinstance(lab2, float) else str(lab2)
    rows.append((f"Weed recall by size — {lab}", val_str))

df_summary = pd.DataFrame(rows, columns=["Metric", "Value"])
print(df_summary.to_string(index=False))
df_summary.to_csv(f"{CONFIG['SUMMARY_DIR']}/task_metrics_table.csv", index=False)
print("\nTask metrics saved.")

# ── Acceptance criteria check ────────────────────────────────────────────────
print("\n── Acceptance criteria ──────────────────────────────────────────────")
tgt_hr = CONFIG["TARGET_WEED_HIT_RATE_PCT"]
tgt_cd = CONFIG["TARGET_CROP_DAMAGE_RATE_PCT"]
tgt_ms = CONFIG["TARGET_LATENCY_MS"]
print(f"  Weed hit rate @ {NR}px: {hit_rate_nozzle:.1f}% "
      + (f">= {tgt_hr}% ✓" if tgt_hr and hit_rate_nozzle >= tgt_hr
         else (f"< {tgt_hr}% ✗" if tgt_hr else "(target not set — PLACEHOLDER)")))
print(f"  Crop-damage rate     : {crop_damage_rate:.1f}% "
      + (f"<= {tgt_cd}% ✓" if tgt_cd and crop_damage_rate <= tgt_cd
         else (f"> {tgt_cd}% ✗" if tgt_cd else "(target not set — PLACEHOLDER)")))
print(f"  Latency              : (see Section 12)  "
      + (f"target = {tgt_ms} ms" if tgt_ms else "(target not set — PLACEHOLDER)"))


## 11. Failure Analysis

In [ ]:
# ─── 11. Save 20 worst images (missed weeds + crop-damage cases) ──────────────
FAIL_DIR = Path(CONFIG["SUMMARY_DIR"]) / "failure_analysis"
FAIL_DIR.mkdir(exist_ok=True)

# Worst = images with most missed GT weeds (matched=False) or crop-damage hits
df_miss  = df_rec[~df_rec["matched"]].groupby("image").size().reset_index(name="missed_weeds")
df_miss  = df_miss.sort_values("missed_weeds", ascending=False)

def visualise_failure(img_path, gt_sem, gt_inst, preds, fname, out_dir):
    img  = cv2.imread(str(img_path))
    if img is None:
        return
    H, W = img.shape[:2]
    overlay = img.copy()
    # GT weed instances in red
    for iid in np.unique(gt_inst):
        if iid == 0:
            continue
        mask    = (gt_inst == iid).astype(np.uint8)
        maj_sem = Counter(gt_sem[mask == 1].tolist()).most_common(1)[0][0]
        if CONFIG["SEM_TO_YOLO"].get(int(maj_sem), -1) == 1:
            cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.fillPoly(overlay, cnts, (0, 0, 200))
    # predicted weed masks in blue
    if preds.masks is not None:
        for i, ct in enumerate(preds.boxes.cls):
            if int(ct.item()) == 1:
                pmask = (preds.masks.data[i].cpu().numpy() > 0.5).astype(np.uint8)
                if pmask.shape != (H, W):
                    pmask = cv2.resize(pmask, (W, H), interpolation=cv2.INTER_NEAREST)
                cnts2, _ = cv2.findContours(pmask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.fillPoly(overlay, cnts2, (200, 0, 0))
    blended = cv2.addWeighted(img, 0.5, overlay, 0.5, 0)
    cv2.putText(blended, fname[:40], (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.imwrite(str(out_dir / fname), blended)

saved = 0
for _, row in df_miss.head(20).iterrows():
    fname     = row["image"]
    img_path  = OUT / "test" / "images" / fname
    inst_path = GT_ROOT / CONFIG["INSTANCES_DIR"] / fname
    sem_path  = GT_ROOT / CONFIG["SEMANTIC_DIR"]  / fname
    if not inst_path.exists():
        continue
    gt_inst = cv2.imread(str(inst_path), cv2.IMREAD_UNCHANGED)
    gt_sem  = cv2.imread(str(sem_path),  cv2.IMREAD_UNCHANGED)
    preds   = model_inf.predict(
        str(img_path), imgsz=CONFIG["IMGSZ"],
        conf=CONFIG["CONF"], iou=CONFIG["IOU"],
        retina_masks=True, verbose=False,
    )[0]
    try:
        visualise_failure(img_path, gt_sem, gt_inst, preds, fname, FAIL_DIR)
        saved += 1
    except Exception as e:
        print(f"  Skipped {fname}: {e}")

print(f"Saved {saved} failure-analysis images to {FAIL_DIR}")


## 12. Export, Latency Benchmark, Exported-Model Validation

In [ ]:
# ─── 12a. Latency benchmark (PyTorch) ─────────────────────────────────────────
N_WARMUP = 10
N_BENCH  = 50

sample_img = str(next(iter(sorted((OUT / "test" / "images").glob("*.png")))))
model_lat  = YOLO(str(BEST_PT))

print("Warming up…")
for _ in range(N_WARMUP):
    _ = model_lat.predict(sample_img, imgsz=CONFIG["IMGSZ"],
                          conf=CONFIG["CONF"], verbose=False)

print(f"Benchmarking {N_BENCH} inferences…")
latencies = []
for _ in range(N_BENCH):
    t0 = time.perf_counter()
    _  = model_lat.predict(sample_img, imgsz=CONFIG["IMGSZ"],
                           conf=CONFIG["CONF"], verbose=False)
    latencies.append((time.perf_counter() - t0) * 1000)

lat_ms   = np.array(latencies)
perf_res = {
    "hardware": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "imgsz":    CONFIG["IMGSZ"],
    "median_ms": float(np.median(lat_ms)),
    "p95_ms":    float(np.percentile(lat_ms, 95)),
    "fps":       float(1000 / np.median(lat_ms)),
    "model_size_mb": BEST_PT.stat().st_size / 1e6,
}
if torch.cuda.is_available():
    perf_res["peak_gpu_mem_gb"] = torch.cuda.max_memory_allocated() / 1e9

print(f"  Median latency : {perf_res['median_ms']:.1f} ms")
print(f"  P95 latency    : {perf_res['p95_ms']:.1f} ms")
print(f"  FPS            : {perf_res['fps']:.1f}")
print(f"  Model size     : {perf_res['model_size_mb']:.1f} MB")
if "peak_gpu_mem_gb" in perf_res:
    print(f"  Peak GPU mem   : {perf_res['peak_gpu_mem_gb']:.2f} GB")

tgt_ms = CONFIG["TARGET_LATENCY_MS"]
if tgt_ms:
    ok = perf_res["median_ms"] <= tgt_ms
    print(f"  Target {tgt_ms} ms : {'PASS ✓' if ok else 'FAIL ✗'}")
else:
    print("  Latency target: PLACEHOLDER (not set)")


In [ ]:
# ─── 12b. ONNX export ─────────────────────────────────────────────────────────
print("Exporting to ONNX…")
try:
    onnx_path = model_best.export(format="onnx", imgsz=CONFIG["IMGSZ"])
    print(f"ONNX model: {onnx_path}")
    perf_res["onnx_path"] = str(onnx_path)
except Exception as e:
    print(f"ONNX export failed: {e}")
    onnx_path = None

# ─── 12c. TensorRT FP16 export (if available) ─────────────────────────────────
trt_path = None
try:
    if torch.cuda.is_available():
        print("Exporting to TensorRT FP16…")
        trt_path = model_best.export(format="engine", imgsz=CONFIG["IMGSZ"], half=True)
        print(f"TensorRT engine: {trt_path}")
        perf_res["trt_path"] = str(trt_path)
    else:
        print("TensorRT export skipped — no CUDA GPU.")
except Exception as e:
    print(f"TensorRT export failed or not available: {e}")

# ─── 12d. Validate exported model (ONNX) ──────────────────────────────────────
if onnx_path:
    try:
        model_onnx = YOLO(str(onnx_path))
        onnx_val   = model_onnx.val(
            data=str(yaml_path), split="test",
            imgsz=CONFIG["IMGSZ"], conf=CONFIG["CONF"], iou=CONFIG["IOU"],
            retina_masks=CONFIG["RETINA_MASKS"], verbose=False,
        )
        delta_map50 = onnx_val.seg.map50 - test_results.seg.map50
        print(f"\nONNX vs PyTorch — seg mAP@0.5 delta: {delta_map50:+.4f}")
        if abs(delta_map50) > 0.02:
            print("WARNING: large mAP delta (>0.02) — check export settings.")
        else:
            print("Delta within acceptable range (≤ 0.02).")
        perf_res["onnx_seg_map50"] = float(onnx_val.seg.map50)
        perf_res["onnx_vs_pt_delta_map50"] = float(delta_map50)
    except Exception as e:
        print(f"ONNX validation failed: {e}")

# ─── 12e. Validate TensorRT model ─────────────────────────────────────────────
if trt_path:
    try:
        model_trt = YOLO(str(trt_path))
        trt_val   = model_trt.val(
            data=str(yaml_path), split="test",
            imgsz=CONFIG["IMGSZ"], conf=CONFIG["CONF"], iou=CONFIG["IOU"],
            retina_masks=CONFIG["RETINA_MASKS"], verbose=False,
        )
        delta_trt = trt_val.seg.map50 - test_results.seg.map50
        print(f"TensorRT FP16 vs PyTorch — seg mAP@0.5 delta: {delta_trt:+.4f}")
        perf_res["trt_seg_map50"]         = float(trt_val.seg.map50)
        perf_res["trt_vs_pt_delta_map50"] = float(delta_trt)
    except Exception as e:
        print(f"TensorRT validation failed: {e}")

with open(f"{CONFIG['SUMMARY_DIR']}/perf_metrics.json", "w") as f:
    json.dump(perf_res, f, indent=2, default=str)
print("\nPerformance metrics saved.")


WARNING ⚠️ Retry 1/2 failed: Command '['/arc/project/st-sielmann-1/envs/weed/bin/python', '-m', 'pip', 'install', '--no-cache-dir', 'onnx>=1.12.0,<2.0.0', 'onnxruntime-gpu', 'onnxslim>=0.1.82']' returned non-zero exit status 1.


## 13. Save Summary (Weights, Config, Metrics)

In [ ]:
# ─── 13. Collect everything and save ─────────────────────────────────────────
summary = {
    "config": {k: str(v) for k, v in CONFIG.items()},
    "best_weights": str(BEST_PT),
    "run_dir": str(RUN_DIR),
    "standard_metrics_val":  std_metrics,
    "standard_metrics_test": test_std_metrics,
    "task_metrics":          task_metrics,
    "performance":           perf_res,
}
summ_json = Path(CONFIG["SUMMARY_DIR"]) / "summary.json"
with open(summ_json, "w") as f:
    json.dump(summary, f, indent=2, default=str)

summ_csv = Path(CONFIG["SUMMARY_DIR"]) / "summary_headline.csv"
rows = [
    ("model",             CONFIG["MODEL"]),
    ("best_weights",      str(BEST_PT)),
    ("seg_map50_val",     std_metrics["seg_map50"]),
    ("seg_map50_test",    test_std_metrics["seg_map50"]),
    ("seg_map_test",      test_std_metrics["seg_map"]),
    ("weed_recall_test",  test_std_metrics["per_class"]["weed"]["recall"]),
    ("crop_recall_test",  test_std_metrics["per_class"]["crop"]["recall"]),
    ("hit_rate_mask_pct", task_metrics["hit_rate_in_mask_pct"]),
    ("hit_rate_nozzle_pct", task_metrics["hit_rate_in_nozzle_pct"]),
    ("crop_damage_rate_pct", task_metrics["crop_damage_rate_pct"]),
    ("false_spray_rate_pct", task_metrics["false_spray_rate_pct"]),
    ("latency_median_ms", perf_res.get("median_ms","n/a")),
    ("latency_p95_ms",    perf_res.get("p95_ms","n/a")),
    ("conf_threshold",    CONFIG["CONF"]),
    ("iou_threshold",     CONFIG["IOU"]),
]
pd.DataFrame(rows, columns=["metric","value"]).to_csv(summ_csv, index=False)

print("Summary saved:")
print(f"  JSON : {summ_json}")
print(f"  CSV  : {summ_csv}")
print("\nFiles in results/:")
for f in sorted(Path(CONFIG["SUMMARY_DIR"]).glob("*")):
    print(f"  {f.name}")


## Final Report

*(Fill in actual measured values after running all cells.)*

| Item | Value |
|------|-------|
| Model | `CONFIG["MODEL"]` |
| Image size | 1024 × 1024 |
| Internal val seg mAP@0.5 | — |
| Test seg mAP@0.5 | — |
| Test seg mAP@0.5:0.95 | — |
| Weed recall (test) | — |
| Crop recall (test) | — |
| Hit rate — inside GT mask | — |
| Hit rate — within NOZZLE_RADIUS_PX | — |
| Crop-damage rate | — |
| False-spray rate | — |
| Latency median / P95 | — ms / — ms |
| Confidence threshold | — |

### Operating Point

The confidence threshold was chosen to maximise weed recall (weed-F1 ≥ 90 % of peak)
while keeping the mean F1 as high as possible.
This conservative policy favours **not missing weeds** over avoiding false positives,
which is appropriate for a sprayer where a missed weed persists.

### Main Failure Modes *(to be filled after failure analysis)*

- Tiny weeds (< 100 px²) are likely to be missed — consider a dedicated small-object head or a higher-resolution crop.
- Partial weeds at image borders may generate spurious polygons.
- Similar leaf texture between crop and weed may cause class confusion in dense canopies.

### Recommended Next Steps

1. Collect a small fine-tune set (200–500 images) from the **real sprayer camera** at operational height.
   Use SAM-assisted annotation to speed up labelling.
2. Validate that GSD (mm/px) from the production camera matches PhenoBench assumptions;
   update `CONFIG["GSD_MM_PX"]` and re-report center error in mm.
3. Set real acceptance targets (`TARGET_WEED_HIT_RATE_PCT`, `TARGET_CROP_DAMAGE_RATE_PCT`,
   `TARGET_LATENCY_MS`) with the agronomist and re-run the acceptance check.
4. Try `yolo11m-seg.pt` for potentially better small-weed recall; compare latency trade-off.
5. Consider TensorRT FP16 for real-time deployment if latency is the bottleneck.

> **Caution:** These results are specific to PhenoBench (one camera, one field, specific growth stages).
> Do not assume they transfer to a different camera, height, crop, or phenology without re-validation.
